# Traces → SFT distillationThis notebook distils a deployed Foundry agent's real tool-using behavior into a smaller, cheaper student model — using **real production traces** as training data.**You'll end up with**: a `gpt-4.1-nano` (or other small model) fine-tuned to call the same tools, in the same order, with the same arguments as your hosted agent — at roughly an order of magnitude lower inference cost per token.**Time**: ~30–50 minutes (most of it waiting for FT jobs).**Cost**: ~$3–8 per run.---## What you need1. **A deployed Foundry hosted agent** with **historical traces** in App Insights. Even ~50–100 real conversations is enough to distill into a usable model.   - If you don't have traces yet, run `fixtures/push_prompts.py` against your agent first (the script pushes 500 diverse retail prompts; wait ~90 seconds for traces to land).2. **The agent's system prompt + tool catalog** as separate files. (Foundry's trace export currently does not emit these at the row level, so you supply them — see `fixtures/zava_system_prompt.md` and `fixtures/zava_tools.json` for the included Zava example.)3. Environment variables:   - `AZURE_AI_PROJECT_ENDPOINT` — the project that hosts your agent   - `OPENAI_BASE_URL` — the Azure OpenAI endpoint with FT support   - `AZURE_OPENAI_API_KEY` — key for the OpenAI endpoint4. `FINETUNING_SKILL_PATH` — path to the `microsoft-foundry/fine-tuning` skill's `Skills/` folder.

In [ ]:
import os, json, subprocess, sysfrom pathlib import PathPROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]BASE_URL         = os.environ["OPENAI_BASE_URL"]API_KEY          = os.environ["AZURE_OPENAI_API_KEY"]SKILL            = Path(os.environ.get("FINETUNING_SKILL_PATH", "../../../Skills"))WORK             = Path("./run").resolve()WORK.mkdir(exist_ok=True)AGENT_NAME    = "demo1-retail-agent-langraph-responses"   # ← REPLACE with your hosted agent nameAGENT_VERSION = "5"                                       # ← REPLACE with the version you want traces fromHOURS_LOOKBACK = 720                                      # last 30 daysprint(f"Project:   {PROJECT_ENDPOINT}")print(f"Agent:     {AGENT_NAME}:{AGENT_VERSION}")print(f"Lookback:  {HOURS_LOOKBACK}h")print(f"Work dir:  {WORK}")

## (Optional) Push fresh prompts through your agentSkip this cell if your agent already has trace history. Otherwise run the bundled `push_prompts.py` which sends ~500 diverse retail-style queries through your agent. The traces land in App Insights within ~90 seconds.

In [ ]:
SKIP_PROMPT_PUSH = True   # set to False if your agent needs fresh trace historyif not SKIP_PROMPT_PUSH:    subprocess.run([        sys.executable, "fixtures/push_prompts.py",        "--agent-name", AGENT_NAME,        "--agent-version", AGENT_VERSION,        "--num-prompts", "500",        "--concurrency", "4",        "--project-endpoint", PROJECT_ENDPOINT,    ], check=True)    print("\nWaiting 90s for traces to land in App Insights...")    import time; time.sleep(90)else:    print("Skipping fresh prompt push — using existing trace history.")

## 1. Run the distillation autopilotThis shells out to the skill's `auto_finetune.py auto` with:- `--datagen-backend foundry-traces` — pull conversations from App Insights for the named agent- `--traces-system-prompt-file` + `--traces-tools-file` — inject the agent's system prompt and tool catalog into each training row (Phase 2a transform)- Three default candidates: `conservative` (3ep lr=1.0), `high-lr` (3ep lr=2.0), `alt-mini` (gpt-4.1-mini 3ep lr=1.0)The autopilot's Phase 2a wiring is the key piece — it shells out to `transform_traces_jsonl.py` automatically, which fixes five upstream issues in the trace export that would otherwise cause Azure FT preprocessing to reject the data.

In [ ]:
STUDENT_MODEL = "gpt-4.1-nano"  # the small model we're distilling INTOTASK_NAME     = "zava-distil"cmd = [    sys.executable, str(SKILL / "scripts" / "auto_finetune.py"), "auto",    "--description", "Distil the deployed Zava Post-Purchase Resolution Desk agent into a smaller "                     "model that handles the same queries with the same tools and style.",    "--task-name", TASK_NAME,    "--model", STUDENT_MODEL,    "--datagen-backend", "foundry-traces",    "--datagen-agent-name", AGENT_NAME,    "--datagen-agent-version", AGENT_VERSION,    "--datagen-hours", str(HOURS_LOOKBACK),    "--traces-system-prompt-file", str(Path("fixtures/zava_system_prompt.md").resolve()),    "--traces-tools-file", str(Path("fixtures/zava_tools.json").resolve()),    "--num-examples", "100",    "--max-iterations", "1",    "--max-budget", "15",    "--work-dir", str(WORK),    "--tier", "globalStandard",    "--project-endpoint", PROJECT_ENDPOINT,    "--base-url", BASE_URL,    "--api-key", API_KEY,]proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)for line in proc.stdout:    print(line, end="")proc.wait()print(f"\n[exit {proc.returncode}]")

## 2. Inspect the leaderboardThe autopilot writes `review_iter1.json` with the SHIP/ITERATE/STOP decision and per-candidate scores.

In [ ]:
review = json.loads((WORK / "review_iter1.json").read_text())print(f"DECISION: {review['decision']}")print(f"REASON  : {review.get('reason', '(none)')}")print()print("Per-candidate leaderboard:")print(f"  {'candidate':<25} {'combined':>9} {'pass_rate':>10} {'lift_vs_base':>13}")print(f"  {'-'*25} {'-'*9} {'-'*10} {'-'*13}")for c in review.get("candidates", []):    lift = c.get("lift_pct")    lift_s = f"{lift:+.1f}%" if lift is not None else "—"    print(f"  {c['candidate']:<25} {c.get('combined', 0):>9.2f} {c.get('pass_rate', 0):>9.1f}% {lift_s:>13}")print()if review["decision"] == "SHIP":    print(f"✅ Shipped model: {review.get('winner', {}).get('model_id', '?')}")

## 3. Try the fine-tuned studentIf a candidate shipped, you can call it with `tools=` enabled — it should emit tool_calls the same way the teacher agent does, at ~10× lower cost per token.

In [ ]:
if review["decision"] != "SHIP":    print("Skipping inference test — no winner this run.")else:    from openai import OpenAI    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)    winner_model = review["winner"]["model_id"]    tools = json.loads(Path("fixtures/zava_tools.json").read_text())    system_prompt = Path("fixtures/zava_system_prompt.md").read_text()    test_prompt = ("My order #ZA-2057 from 3 weeks ago — the wireless headphones arrived but they "                   "stopped working after a few days. I'd like a refund.")    resp = client.chat.completions.create(        model=winner_model,        messages=[            {"role": "system", "content": system_prompt},            {"role": "user", "content": test_prompt},        ],        tools=tools,        temperature=0,    )    msg = resp.choices[0].message    print(f"Distilled model: {winner_model}")    print(f"Tool calls emitted:")    for tc in (msg.tool_calls or []):        print(f"  {tc.function.name}({tc.function.arguments})")    if not msg.tool_calls:        print(f"  (no tool calls; text: {(msg.content or '')[:200]})")

## Notes- **Eval signal**: when ref tool_calls are present on a test row, the autopilot's `_evaluate_model_on_test` runs inference WITH `tools=` enabled and scores by **structural match** (tool name + arguments). Falls back to text-judge for text-only rows. The eval coverage breakdown is reported in the autopilot output.- **Why "100 examples"**: the foundry-traces recipe pulls real conversations; you need at minimum ~50 to get a meaningful train/val/test split. 100-300 is sweet-spot for most agents. Past 500 and you may need to revisit the `--max-budget` flag because eval cost grows with test set size.- **Bring your own agent**: replace `AGENT_NAME` / `AGENT_VERSION` and the two fixture files with your agent's system prompt + tool catalog. The pipeline handles the rest.- **Upstream**: the 5 transforms in `transform_traces_jsonl.py` exist because Foundry's trace export currently emits chat JSONL that needs fixing for tool-using FT. These should disappear as the export improves; the skill will continue to work either way.